# **Build a custom Question Answering (QA) system using BERT**

In [1]:
# !pip install transformers datasets torch

In [2]:
%%writefile custom_qa_dataset.json

[
  {
    "context": "BERT is a transformer-based model designed to perform NLP tasks such as question answering, language understanding, and text classification.",
    "question": "What is BERT?",
    "answers": {
      "text": ["a transformer-based model"],
      "answer_start": [10]
    }
  },
  {
    "context": "Hugging Face provides pre-trained transformers for various NLP tasks.",
    "question": "What does Hugging Face provide?",
    "answers": {
      "text": ["pre-trained transformers"],
      "answer_start": [14]
    }
  }
]


Writing custom_qa_dataset.json


In [5]:
#load it using Hugging Face's datasets library
from datasets import load_dataset

# Load your custom dataset
dataset = load_dataset('json', data_files='custom_qa_dataset.json')
dataset

DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'answers'],
        num_rows: 2
    })
})

In [6]:

# Split the dataset into train and validation sets (80-20 split)
dataset = dataset['train'].train_test_split(test_size=0.2)
train_dataset = dataset['train']
valid_dataset = dataset['test']


In [7]:
#Preprocess the Dataset
#To prepare the data for training, you need to tokenize the inputs (context + question)
#and map the answers to start and end tokens in the context.

from transformers import BertTokenizerFast

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

def prepare_features(examples):
    # Tokenize the inputs (context + question)
    tokenized_examples = tokenizer(
        examples['question'],
        examples['context'],
        truncation="only_second",  # truncate context if too long
        max_length=384,  # maximum input length for BERT
        stride=128,  # overlap between segments if the context is too long
        return_overflowing_tokens=True,  # handle large contexts
        return_offsets_mapping=True,  # provide token position offsets
        padding="max_length"
    )

    # Find the position of the start and end tokens of the answer
    offset_mapping = tokenized_examples.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # Find where the answer starts and ends in the context
        answer = examples['answers'][i]
        start_char = answer['answer_start'][0]
        end_char = start_char + len(answer['text'][0])

        # Find token start and end positions
        token_start = 0
        token_end = 0
        for idx, offset in enumerate(offsets):
            if offset[0] <= start_char and offset[1] >= start_char:
                token_start = idx
            if offset[0] <= end_char and offset[1] >= end_char:
                token_end = idx

        start_positions.append(token_start)
        end_positions.append(token_end)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions
    return tokenized_examples

# Apply the preprocessing function
train_dataset = train_dataset.map(prepare_features, batched=True, remove_columns=["context", "question", "answers"])
valid_dataset = valid_dataset.map(prepare_features, batched=True, remove_columns=["context", "question", "answers"])


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [8]:
# Load the Pre-trained BERT Model
# You can fine-tune the bert-base-uncased model on your custom QA task. The
# BertForQuestionAnswering class is specifically designed for this:

from transformers import BertForQuestionAnswering

# Load the BERT model for question answering
model = BertForQuestionAnswering.from_pretrained('bert-base-uncased')


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
#Train the Model
#Now, define your training arguments and train the model using the Trainer class.

from transformers import Trainer, TrainingArguments

# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    evaluation_strategy="epoch",     # evaluation strategy
    learning_rate=3e-5,              # learning rate
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    num_train_epochs=3,              # number of training epochs
    weight_decay=0.01,               # weight decay
)

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
)

# Train the model
trainer.train()


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
2024-09-17 12:49:34.685793: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/anaconda3/lib/python3.11/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 6.3188886642456055, 'eval_runtime': 0.3348, 'eval_samples_per_second': 2.987, 'eval_steps_per_second': 2.987, 'epoch': 1.0}


  0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 6.265023231506348, 'eval_runtime': 0.3426, 'eval_samples_per_second': 2.919, 'eval_steps_per_second': 2.919, 'epoch': 2.0}


  0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 6.2398247718811035, 'eval_runtime': 0.3397, 'eval_samples_per_second': 2.944, 'eval_steps_per_second': 2.944, 'epoch': 3.0}
{'train_runtime': 10.8218, 'train_samples_per_second': 0.277, 'train_steps_per_second': 0.277, 'train_loss': 5.417374928792317, 'epoch': 3.0}


TrainOutput(global_step=3, training_loss=5.417374928792317, metrics={'train_runtime': 10.8218, 'train_samples_per_second': 0.277, 'train_steps_per_second': 0.277, 'total_flos': 587917702656.0, 'train_loss': 5.417374928792317, 'epoch': 3.0})

In [7]:
#Evaluate the Model
#You can evaluate the model using the evaluate() method provided by the Trainer class.

# Evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")


Evaluation Results: {'eval_loss': 5.88313102722168, 'eval_runtime': 2.6615, 'eval_samples_per_second': 0.376, 'eval_steps_per_second': 0.376, 'epoch': 3.0}


In [10]:
import torch

# Provide a new context and question
context = "BERT is a transformer-based model designed to perform NLP tasks."
question = "What is BERT?"

# Tokenize the input
inputs = tokenizer(question, context, return_tensors="pt")

# Get model predictions
with torch.no_grad():
    outputs = model(**inputs)
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

# Get the most likely beginning and end of the answer
start_index = torch.argmax(start_logits)
end_index = torch.argmax(end_logits)

# Convert token IDs back to the answer
answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start_index:end_index+1]))
print(f"Answer: {answer}")


Answer: transformer - based model designed
